## Description

In this programming assignment, you are required to implement the Apriori algorithm and apply it to mine frequent itemsets from a real-life data set.

**Input**

The provided input file ("categories.txt") consists of the category lists of 77,185 places in the US. Each line corresponds to the category list of one place, where the list consists of a number of category instances (e.g., hotels, restaurants, etc.) that are separated by semicolons.

An example line is provided below:

Local Services;IT Services & Computer Repair

In the example above, the corresponding place has two category instances: "Local Services" and "IT Services & Computer Repair".

**Output**

You need to implement the Apriori algorithm and use it to mine category sets that are frequent in the input data. When implementing the Apriori algorithm, you may use any programming language you like. We only need your result pattern file, not your source code file.

After implementing the Apriori algorithm, please set the relative minimum support to 0.01 and run it on the 77,185 category lists. In other words, you need to extract all the category sets that have an absolute support larger than 771.

**Part 1**

Please output all the length-1 frequent categories with their absolute supports into a text file named "patterns.txt". Every line corresponds to exactly one frequent category and should be in the following format:

support:category

For example, suppose a category (Fast Food) has an absolute support 3000, then the line corresponding to this frequent category set in "patterns.txt" should be:

3000:Fast Food

Part 2

Please write all the frequent category sets along with their absolute supports into a text file named "patterns.txt". Every line corresponds to exactly one frequent category set and should be in the following format:

support:category_1;category_2;category_3;...

For example, suppose a category set (Fast Food; Restaurants) has an absolute support 2851, then the line corresponding to this frequent category set in "patterns.txt" should be:

2851:Fast Food;Restaurants

Important Tips
Make sure that you format each line correctly in the output file. For instance, use a semicolon instead of another character to separate the categories for each frequent category set. 

In the result pattern file, the order of the categories does not matter. For example, the following two cases will be considered equivalent by the grader:

Case 1:

2851:Fast Food;Restaurants

Case 2:

2851:Restaurants;Fast Food 

In [1]:
from collections import Counter
from itertools import combinations
from math import ceil
from pathlib import Path

DATA_PATH = Path("categories.txt")
PART1_OUTPUT_PATH = Path("patterns_part1.txt")
PART2_OUTPUT_PATH = Path("patterns.txt")
MIN_SUPPORT_RATIO = 0.01


def load_transactions(path: Path):
    transactions = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            items = {item.strip() for item in line.split(";") if item.strip()}
            if items:
                transactions.append(frozenset(items))
    return transactions


def frequent_1_itemsets(transactions, min_support_abs: int):
    counter = Counter()
    for transaction in transactions:
        counter.update(transaction)
    return {frozenset([item]): support for item, support in counter.items() if support >= min_support_abs}


def generate_candidates(prev_level_itemsets, k: int):
    prev_level_list = sorted(prev_level_itemsets, key=lambda s: tuple(sorted(s)))
    prev_level_set = set(prev_level_itemsets)
    candidates = set()

    for i in range(len(prev_level_list)):
        for j in range(i + 1, len(prev_level_list)):
            merged = prev_level_list[i] | prev_level_list[j]
            if len(merged) != k:
                continue
            # Apriori pruning: all (k-1)-subsets of a candidate must be frequent.
            if all(frozenset(subset) in prev_level_set for subset in combinations(merged, k - 1)):
                candidates.add(frozenset(merged))

    return candidates


def count_candidate_support(candidates, transactions, k: int, min_support_abs: int):
    candidate_set = set(candidates)
    support_counter = Counter()

    for transaction in transactions:
        if len(transaction) < k:
            continue
        for subset in combinations(sorted(transaction), k):
            subset_frozen = frozenset(subset)
            if subset_frozen in candidate_set:
                support_counter[subset_frozen] += 1

    return {itemset: support for itemset, support in support_counter.items() if support >= min_support_abs}


def apriori(transactions, min_support_abs: int):
    all_frequent = {}
    level_frequent = frequent_1_itemsets(transactions, min_support_abs)
    all_frequent.update(level_frequent)

    k = 2
    current_level_itemsets = set(level_frequent.keys())

    while current_level_itemsets:
        candidates = generate_candidates(current_level_itemsets, k)
        if not candidates:
            break

        level_frequent = count_candidate_support(candidates, transactions, k, min_support_abs)
        if not level_frequent:
            break

        all_frequent.update(level_frequent)
        current_level_itemsets = set(level_frequent.keys())
        k += 1

    return all_frequent


def itemset_to_text(itemset):
    return ";".join(sorted(itemset))


def write_patterns(output_path: Path, frequent_itemsets):
    sorted_patterns = sorted(
        frequent_itemsets.items(),
        key=lambda x: (-x[1], len(x[0]), itemset_to_text(x[0]))
    )
    with output_path.open("w", encoding="utf-8") as file:
        for itemset, support in sorted_patterns:
            file.write(f"{support}:{itemset_to_text(itemset)}\n")


transactions = load_transactions(DATA_PATH)
min_support_abs = ceil(MIN_SUPPORT_RATIO * len(transactions))

all_frequent_itemsets = apriori(transactions, min_support_abs)
length_1_itemsets = {itemset: support for itemset, support in all_frequent_itemsets.items() if len(itemset) == 1}

write_patterns(PART1_OUTPUT_PATH, length_1_itemsets)
write_patterns(PART2_OUTPUT_PATH, all_frequent_itemsets)

print(f"Transactions: {len(transactions)}")
print(f"Min support ratio: {MIN_SUPPORT_RATIO}")
print(f"Min support absolute: {min_support_abs}")
print(f"Frequent 1-itemsets: {len(length_1_itemsets)}")
print(f"All frequent itemsets: {len(all_frequent_itemsets)}")
print(f"Part 1 output: {PART1_OUTPUT_PATH.resolve()}")
print(f"Part 2 output: {PART2_OUTPUT_PATH.resolve()}")

Transactions: 77185
Min support ratio: 0.01
Min support absolute: 772
Frequent 1-itemsets: 50
All frequent itemsets: 101
Part 1 output: /Users/vananhduy/Documents/Repository_Git_Hub/SP26/DBM302m/coursera labs/Pattern Discovery in Data Mining/patterns_part1.txt
Part 2 output: /Users/vananhduy/Documents/Repository_Git_Hub/SP26/DBM302m/coursera labs/Pattern Discovery in Data Mining/patterns.txt
